# Amazon Redshift Fundamentals & Data Warehousing
This notebook covers the basic concepts of Amazon Redshift for data engineering. 

### Learning Objectives:
1. Connect to Redshift from Python.
2. Execute DDL and DML queries.
3. Monitor table health (unsorted data & stale statistics).
4. Inspect query execution plans.

## 1. Import Packages & Connect to Redshift
We use our custom `RedshiftConnector` which defaults to a mock connection if no live credentials are found in the configuration.

In [ ]:
import sys
import os

# Ensure our src directory is in the path
sys.path.append(os.path.abspath('../'))

from src.redshift_connector import RedshiftConnector
from src.redshift_utils import RedshiftUtils

# Instantiate the connector pointing to our yaml config
config_path = '../configs/redshift_config.yaml'
connector = RedshiftConnector(config_path)
conn = connector.get_connection()
utils = RedshiftUtils(conn)
print(f"Connected in Mock Mode? {connector.mock_mode}")

## 2. Querying Table Health Metadata
We will query the Redshift system catalog view `svv_table_info` to get metrics such as row count, size, unsorted percent, and stats off percent.

In [ ]:
health_stats = utils.get_table_health()
print("=== REDSHIFT TABLE HEALTH STATUS ===")
for table in health_stats:
    print(f"Table: {table['schema']}.{table['table']}")
    print(f"  Rows: {table['rows']:,}")
    print(f"  Size: {table['size_mb']} MB")
    print(f"  Unsorted: {table['unsorted_pct']}%")
    print(f"  Stats Off: {table['stats_off_pct']}%")
    print(f"  Vacuum Sort Benefit: {table['vacuum_sort_benefit']}")
    print("-" * 40)

## 3. Simulating Database Maintenance
We can trigger manual `VACUUM` and `ANALYZE` statements if the statistics are stale or data is highly unsorted.

In [ ]:
cursor = conn.cursor()
try:
    print("Running ANALYZE on core.fact_sales...")
    cursor.execute("ANALYZE core.fact_sales;")
    print("Running VACUUM SORT ONLY on core.fact_sales...")
    cursor.execute("VACUUM SORT ONLY core.fact_sales;")
    conn.commit()
    print("Maintenance completed successfully.")
except Exception as e:
    conn.rollback()
    print(f"Failed to perform maintenance: {e}")
finally:
    cursor.close()

## 4. Explaining a Join Query
Let's print the execution plan of a query joining `fact_sales` and `dim_customer` on `customer_id`.

In [ ]:
explain_query = """
EXPLAIN
SELECT 
    c.country,
    SUM(s.quantity * s.unit_price) AS total_sales
FROM core.fact_sales s
INNER JOIN core.dim_customer c ON s.customer_id = c.customer_id
GROUP BY c.country;
"""

try:
    plan_steps = utils.execute_sql(explain_query)
    print("=== QUERY PLAN ===")
    for step in plan_steps:
        print(step[0])
except Exception as e:
    print(f"Could not run EXPLAIN query: {e}")